In [ ]:
from binn import BINN, BINNDataLoader, BINNTrainer, BINNExplainer
import pandas as pd

import textwrap
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pandas as pd

from binn.plot.util import (
    load_default_mapping,
    build_mapping_dict,
    rename_node_by_layer
)

def visualize_binn(
    dataframe,
    top_n=5,
    layer_normalization_value=1,
    sink_node_size=500,
    sink_node_color="lightgray",
    node_cmap="viridis",
    plot_size=(12, 8),
    alpha=0.9,
    edge_color="gray",
    edge_alpha=0.7,
    arrow_size=10,
    edge_width=1,
    node_size_scaling=10,
    input_entity_mapping=None,  # can be a str, a DataFrame, or None
    pathways_mapping=None,      # can be a str, a DataFrame, or None
    input_entity_key_col="input_id",      # which column to use as key if user-provided
    input_entity_val_col="input_name",       # which column to use as value if user-provided
    pathways_key_col="pathway_id",    # which column to use as key if user-provided
    pathways_val_col="pathway_name",           # which column to use as value if user-provided
):
    """
    Visualizes the most important nodes in a network using a multipartite (layered) layout,
    with a single output node to represent all non-top connections in the final layer.

    'Sink nodes' gather all non-top nodes from a given layer and inherit their connections.
    They are drawn at the bottom of each layer with fixed size and color, and are excluded
    from the colormap scaling.

    Parameters:
    ----------
    dataframe : pd.DataFrame
        Must have columns: 
            [source_node, target_node, source_layer, target_layer, normalized_importance].
    top_n : int or dict
        Number of top nodes to visualize per layer. 
        If an int, applies to all layers. If dict, keys are layers and values are ints.
    layer_normalization_value : float
        Normalization value for importance in each layer.
    sink_node_size : float
        Fixed size for sink nodes.
    sink_node_color : str
        Color for sink nodes.
    node_cmap : str
        Colormap name for non-sink nodes.
    plot_size : tuple
        (width, height) for the figure.
    alpha : float
        Transparency for node colors.
    edge_color : str
        Color for edges.
    edge_alpha : float
        Transparency for edges.
    arrow_size : int
        Size of arrows in the drawn edges.
    edge_width : float
        Width of edges.
    node_size_scaling : float
        Scaling factor for node size based on importance.

    input_entity_mapping : {None, str, pd.DataFrame}
        If None, no mapping is done for layer 1.
        If str (e.g. "reactome"), we attempt to load a default DataFrame 
        from the package (not typical for input entities, but flexible).
        If pd.DataFrame, we build a dictionary for renaming.

    pathways_mapping : {None, str, pd.DataFrame}
        If None, no mapping is done for layers != 1.
        If str (e.g. "reactome"), we attempt to load a default DataFrame 
        from the package.
        If pd.DataFrame, we build a dictionary for renaming.

    input_entity_key_col : str
        Column name used as the key in input_entity_mapping if it's a DataFrame.
    input_entity_val_col : str
        Column name used as the value in input_entity_mapping.

    pathways_key_col : str
        Column name used as the key in pathways_mapping if it's a DataFrame.
    pathways_val_col : str
        Column name used as the value in pathways_mapping.
    """

    # ---------------------------------------------------------------------
    # 1) Build or load dictionaries for input entities and pathways
    # ---------------------------------------------------------------------
    # input entities (usually for layer 1)
    if isinstance(input_entity_mapping, str):
        # load default df, then build dict
        input_entity_df = load_default_mapping(input_entity_mapping)
        input_entity_dict = build_mapping_dict(
            input_entity_df, key_col=input_entity_key_col, val_col=input_entity_val_col
        )
    elif isinstance(input_entity_mapping, pd.DataFrame):
        # user-provided df => build dict
        input_entity_dict = build_mapping_dict(
            input_entity_mapping, key_col=input_entity_key_col, val_col=input_entity_val_col
        )
    else:
        # None or anything else => no mapping
        input_entity_dict = None

    # pathways (usually for layers != 1)
    if isinstance(pathways_mapping, str):
        # load default df, then build dict
        pathways_df = load_default_mapping(pathways_mapping)
        pathways_dict = build_mapping_dict(
            pathways_df, key_col=pathways_key_col, val_col=pathways_val_col
        )
    elif isinstance(pathways_mapping, pd.DataFrame):
        # user-provided df => build dict
        pathways_dict = build_mapping_dict(
            pathways_mapping, key_col=pathways_key_col, val_col=pathways_val_col
        )
    else:
        pathways_dict = None

    # ---------------------------------------------------------------------
    # 2) Preprocess the main dataframe: ensure layers are strings, build (node, layer) tuples
    # ---------------------------------------------------------------------
    dataframe["source_layer"] = dataframe["source_layer"].astype(str)
    dataframe["target_layer"] = dataframe["target_layer"].astype(str)

    dataframe["source_tuple"] = list(zip(dataframe["source_node"], dataframe["source_layer"]))
    dataframe["target_tuple"] = list(zip(dataframe["target_node"], dataframe["target_layer"]))

    # ---------------------------------------------------------------------
    # 3) Aggregate + normalize node importance
    # ---------------------------------------------------------------------
    if "normalized_importance" in dataframe.columns:
        value_column = "normalized_importance"
    else:
        value_column = "importance"

    source_importance_df = (
        dataframe
        .groupby(["source_tuple", "source_layer"])[value_column]
        .sum()
        .reset_index()
    )

    # Normalize importance by layer
    source_importance_df[value_column] = source_importance_df.groupby("source_layer")[
        value_column
    ].transform(lambda x: x / x.sum() * layer_normalization_value)

    # Handle top_n as dict
    if isinstance(top_n, int):
        layers_unique = source_importance_df["source_layer"].unique()
        top_n = {layer: top_n for layer in layers_unique}

    def pick_top(group):
        n = top_n.get(group.name, 0)
        return group.nlargest(n, value_column)

    top_nodes_by_layer = (
        source_importance_df
        .groupby("source_layer")
        .apply(pick_top)
        .reset_index(drop=True)
    )
    top_source_nodes = set(top_nodes_by_layer["source_tuple"])

    # ---------------------------------------------------------------------
    # 4) Identify sink nodes
    # ---------------------------------------------------------------------
    all_layers = set(dataframe["source_layer"].unique()).union(
        set(dataframe["target_layer"].unique())
    )
    final_layer = max(int(layer) for layer in all_layers)

    # We'll call them ("sink", layer) except for final layer => ("output_node", final_layer)
    sink_nodes = {}
    for layer_str in all_layers:
        if int(layer_str) == final_layer:
            sink_nodes[layer_str] = ("AD dementia diagnosis", layer_str)
        else:
            sink_nodes[layer_str] = ("sink", layer_str)

    # ---------------------------------------------------------------------
    # 5) Build the directed graph
    # ---------------------------------------------------------------------
    G = nx.DiGraph()
    node_importance = {}

    # initialize top-node importance
    for _, row in top_nodes_by_layer.iterrows():
        node_tuple = row["source_tuple"]
        node_importance[node_tuple] = row[value_column]

    # initialize sink-node importance
    for layer_str, sink_tuple in sink_nodes.items():
        node_importance[sink_tuple] = 0.0

    # For each row in the dataframe, redirect non-top nodes to sink nodes
    for _, row in dataframe.iterrows():
        s_tuple = row["source_tuple"]
        t_tuple = row["target_tuple"]
        s_layer = row["source_layer"]
        t_layer = row["target_layer"]
        importance = row[value_column]

        new_source = s_tuple if s_tuple in top_source_nodes else sink_nodes[s_layer]
        new_target = t_tuple if t_tuple in top_source_nodes else sink_nodes[t_layer]

        if not G.has_node(new_source):
            G.add_node(
                new_source,
                layer=int(s_layer),
                name=rename_node_by_layer(
                    new_source, 
                    input_entity_dict=input_entity_dict, 
                    pathways_dict=pathways_dict
                ),
            )
        if not G.has_node(new_target):
            G.add_node(
                new_target,
                layer=int(t_layer),
                name=rename_node_by_layer(
                    new_target, 
                    input_entity_dict=input_entity_dict, 
                    pathways_dict=pathways_dict
                ),
            )

        node_importance[new_source] = node_importance.get(new_source, 0) + importance
        node_importance[new_target] = node_importance.get(new_target, 0) + importance

        if G.has_edge(new_source, new_target):
            G[new_source][new_target]["weight"] += importance
        else:
            G.add_edge(new_source, new_target, weight=importance)

    # ---------------------------------------------------------------------
    # 6) Figure out node order per layer and generate layout
    # ---------------------------------------------------------------------
    node_layers = {node: data["layer"] for node, data in G.nodes(data=True)}
    layer_nodes = {}
    for node, layer in node_layers.items():
        layer_nodes.setdefault(layer, []).append(node)

    # Sort non-sink nodes within each layer by descending importance
    for layer in layer_nodes:
        sinks_here = [n for n in layer_nodes[layer] if n in sink_nodes.values()]
        non_sink = [n for n in layer_nodes[layer] if n not in sinks_here]
        non_sink_sorted = sorted(non_sink, key=lambda n: -node_importance.get(n, 0))
        layer_nodes[layer] = non_sink_sorted + sinks_here

    pos = nx.multipartite_layout(G, subset_key="layer")

    # Re-center vertically
    for layer, nodes_in_layer in layer_nodes.items():
        count = len(nodes_in_layer)
        offset = (count - 1) / 2.0
        for idx, node in enumerate(nodes_in_layer):
            pos[node] = (layer, offset - idx)

    # ---------------------------------------------------------------------
    # 7) Colormap normalization (exclude sink nodes)
    # ---------------------------------------------------------------------
    cmap = plt.get_cmap(node_cmap)
    layer_norms = {}
    for layer, nodes_in_layer in layer_nodes.items():
        non_sink = [n for n in nodes_in_layer if n not in sink_nodes.values()]
        if non_sink:
            vals = [node_importance[n] for n in non_sink]
            layer_norms[layer] = mcolors.Normalize(vmin=min(vals), vmax=max(vals))
        else:
            layer_norms[layer] = mcolors.Normalize(vmin=0, vmax=1)

    # ---------------------------------------------------------------------
    # 8) Plot
    # ---------------------------------------------------------------------
    plt.figure(figsize=plot_size)
    plt.rcParams['font.size'] = 40
    
    # Draw nodes
    for layer, nodes_in_layer in layer_nodes.items():
        norm = layer_norms[layer]
        sizes = []
        colors = []
        for n in nodes_in_layer:
            if n in sink_nodes.values():
                sizes.append(sink_node_size)
                colors.append(sink_node_color)
            else:
                val = node_importance.get(n, 0)
                sizes.append(val * node_size_scaling)
                colors.append(cmap(norm(val)))
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=nodes_in_layer,
            node_size=sizes,
            node_color=colors,
            alpha=alpha,
        )

    # Draw edges
    nx.draw_networkx_edges(
        G,
        pos,
        arrowstyle="->",
        arrowsize=arrow_size,
        edge_color=edge_color,
        alpha=edge_alpha,
        width=edge_width,
    )

    # Draw labels (the "name" attribute we set above)
    max_label_width = 20
    labels = {
        node: "\n".join(textwrap.wrap(data["name"], width=max_label_width))
        for node, data in G.nodes(data=True)
    }
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=33)

    # Add colorbar
    sm = cm.ScalarMappable(cmap=cmap)
    sm.set_array([])
    plt.colorbar(sm, ax=plt.gca(), label="Normalized node importance", fraction=0.04, pad=0.02)

    plt.axis("off")
    plt.tight_layout()
    return plt

In [ ]:
apoe_identifier = 'APOE2'
sample_datamatrix = pd.read_csv('analysis/clean_withe2e4.csv')
sample_datamatrix = sample_datamatrix[sample_datamatrix['contributor_code'] != 'C']

In [ ]:
if apoe_identifier == 'APOE4':
    sample_datamatrix = sample_datamatrix[sample_datamatrix['APOE'].isin([33, 34, 43, 44])]
    datafolder = 'results/GNPC/e4vse3e3'
    outpath = 'results/GNPC/e4vse3e3/binn.xlsx'
else:
    sample_datamatrix = sample_datamatrix[sample_datamatrix['APOE'].isin([22, 23, 32, 33])]
    datafolder = 'results/GNPC/e2vse3e3'
    outpath = 'results/GNPC/e2vse3e3/binn.xlsx'

In [ ]:
apoe2protein = pd.read_csv(f"{datafolder}/apoe2protein.csv")
apoe2protein_adjad = pd.read_csv(f"{datafolder}/apoe2protein_adjad.csv")
ab2protein = pd.read_csv(f"{datafolder}/ad2protein.csv")
ab2protein_adjapoe = pd.read_csv(f"{datafolder}/ad2protein_adjapoe.csv")
cu_allage = pd.read_csv(f"{datafolder}/cu_allage.csv")

apoe_sig = set(apoe2protein[apoe2protein['p_adjusted'] < 0.05]['Protein_id'].tolist())
apoe_adjab_sig = set(apoe2protein_adjad[apoe2protein_adjad['p_adjusted'] < 0.05]['Protein_id'].tolist())

cu_group_sig = set(cu_allage[cu_allage['p_adjusted'] < 0.05]['Protein_id'].tolist())

In [ ]:
apoe_identifier, len(apoe_sig & cu_group_sig)

In [ ]:
apt2uniprot = dict(zip(apoe2protein['apt_name'].tolist(), apoe2protein['UniProt'].tolist()))
uni2symbol = dict(zip(apoe2protein['UniProt'].tolist(), apoe2protein['symbol'].tolist()))
usedata = apoe2protein[apoe2protein['Protein_id'].isin(apoe_sig & cu_group_sig)]
usedata = usedata.groupby('UniProt', as_index=False).apply(lambda g: g.nsmallest(1, 'p_adjusted')).reset_index(drop=True)
usedata = usedata['apt_name'].tolist()
len(usedata)

In [ ]:
sig_protein_cols = [x for x in sample_datamatrix.columns.tolist() if x[:3] == 'seq' and '.'.join(x.split('_')) in usedata]
len(sig_protein_cols)

In [ ]:
analysis_df = sample_datamatrix[['sample_id', 'diagnosis'] + sig_protein_cols]
analysis_df = analysis_df.dropna(how='any')

In [ ]:
design_matrix=analysis_df[['sample_id', 'diagnosis']]
design_matrix.columns = ['sample', 'group']

In [ ]:
data_matrix = analysis_df[['sample_id'] + sig_protein_cols]
data_matrix.columns = ['sample_id'] + [apt2uniprot['.'.join(x.split('_'))] for x in sig_protein_cols]
data_matrix = data_matrix.set_index('sample_id').T.reset_index()
data_matrix = data_matrix.rename(columns={'index': 'Protein'})

In [ ]:
# Initialize BINN
binn = BINN(data_matrix=data_matrix, network_source="reactome", n_layers=4, dropout=0.2)

## Initialize DataLoader
binn_dataloader = BINNDataLoader(binn)

# Create DataLoaders
dataloaders = binn_dataloader.create_dataloaders(
    data_matrix=data_matrix,
    design_matrix=design_matrix,
    feature_column="Protein",
    group_column="group",
    sample_column="sample",
    batch_size=32,
    validation_split=0.2,
)
# Train the model
trainer = BINNTrainer(binn)
trainer.fit(dataloaders=dataloaders, num_epochs=100)

# Explain the model
explainer = BINNExplainer(binn)
single_explanations = explainer.explain_single(dataloaders, split="val", normalization_method="subgraph")
single_explanations

In [ ]:
single_explanations.to_excel(f'{outpath}', index=False)

In [ ]:
#plt.rcParams['font.size'] = 40
layer_specific_top_n = {"0": 8, "1": 7, "2": 6, "3":5, "4":5}
plt = visualize_binn(single_explanations, top_n=layer_specific_top_n, plot_size=(30,15), sink_node_size=500, node_size_scaling = 1000,
                     edge_width=3,  node_cmap="Reds", pathways_mapping='reactome',
                     input_entity_mapping=pd.DataFrame({'input_id':uni2symbol.keys(), 'input_name':uni2symbol.values()}))#
plt.title(f"BINN-enriched reactome pathways for early {apoe_identifier}-dysregulated proteins", fontsize=35)
plt.savefig(f"Figs/{apoe_identifier}.svg", format='svg', dpi=600, bbox_inches='tight')